# 🔬 Toy Lab Walkthrough

This notebook walks you through the entire mini-project end-to-end:

1. **The toy lab** — what is the hidden response function?
2. **Running experiments** — how do random, grid, and Bayesian strategies work?
3. **Watching beliefs update** — the GP posterior after 5, 10, 20, 30 experiments
4. **Comparing strategies** — learning curves and regret
5. **Key takeaways** — what did we learn about explore/exploit?

Run cells top-to-bottom with **Shift+Enter**.

In [ ]:
# If you haven't installed dependencies yet:
# !pip install -q numpy matplotlib scikit-learn scipy

import sys, os
# Make sure the repo root is on the path
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')   # suppress sklearn convergence chatter

from src.simulator  import LabSimulator
from src.strategies import RandomStrategy, GridStrategy, BayesianStrategy
from src.metrics    import best_found, cumulative_best, simple_regret, summarise
from src.plots      import (
    plot_true_function, plot_observations,
    plot_gp_posterior, plot_learning_curves, plot_posterior_snapshots
)

print('All imports OK ✅')

---
## 1. The toy lab

We have a hidden function f(x) that we want to maximise.  Think of it as the
yield of a chemical reaction at temperature x, or the conversion rate of a
website at font-size x.  We can't see f directly — only noisy measurements.

In [ ]:
# Create the simulator with fixed seed for reproducibility
sim = LabSimulator(noise_std=0.12, bounds=(-2.0, 2.0), rng=42)

fig, ax = plt.subplots(figsize=(9, 4))
plot_true_function(sim, ax=ax)
ax.set_title('The hidden response function (you never see this in a real experiment!)')
plt.tight_layout()
plt.show()

x_opt, f_opt = sim.best_true_value()
print(f'True optimum: f({x_opt:.3f}) = {f_opt:.4f}')

Notice the **bumpy landscape** — there are local peaks and valleys.  A naive
strategy might get stuck in a local peak and never find the global optimum.

The grey band shows ±1σ noise.  Every time we run an experiment at x, the
result lands somewhere in that band.

---
## 2. Running three different strategies

In [ ]:
N_EXPERIMENTS = 30

# Run all three strategies (each resets the simulator history first)
results = {}

for StrategyClass, name, kwargs in [
    (RandomStrategy,   'Random',   {'seed': 0}),
    (GridStrategy,     'Grid',     {}),
    (BayesianStrategy, 'Bayesian', {'n_init': 5, 'kappa': 2.0, 'seed': 0}),
]:
    sim.reset()
    strat = StrategyClass(sim, n_experiments=N_EXPERIMENTS, **kwargs)
    xs, ys = strat.run()
    results[name] = (xs, ys)
    print(f'{name:>10}: best found = {best_found(ys):.4f},  '
          f'simple regret = {simple_regret(xs, ys, sim):.4f}')

In [ ]:
# Show where each strategy placed its experiments
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

colors = {'Random': '#e07b54', 'Grid': '#5b8db8', 'Bayesian': '#3a9e6b'}

for ax, (name, (xs, ys)) in zip(axes, results.items()):
    plot_observations(xs, ys, sim, ax=ax, label=name, color=colors[name])
    ax.set_title(f'{name} strategy ({N_EXPERIMENTS} experiments)')

plt.tight_layout()
plt.show()

**Observation:** Grid search covers the space evenly; random search clusters
in places by chance.  Bayesian search focuses most of its experiments near
the peak — it *learned* where to look.

---
## 3. Watching the GP posterior update

This is the key Bayesian insight: each experiment narrows our uncertainty.
The shaded band (±2σ) shows where we're unsure; it collapses as we gather data.

In [ ]:
# Rebuild the Bayesian run with its own fresh simulator
sim_b = LabSimulator(noise_std=0.12, bounds=(-2.0, 2.0), rng=42)
bayes_strat = BayesianStrategy(sim_b, n_experiments=30, n_init=5, kappa=2.0, seed=0)
xs_b, ys_b = bayes_strat.run()

fig = plot_posterior_snapshots(
    xs_b, ys_b,
    simulator=sim_b,
    strategy=bayes_strat,
    snapshots=(5, 10, 15, 30),
    figsize=(14, 8)
)
plt.show()

After 5 experiments the GP is mostly guessing.  By 30 experiments the mean
(green line) closely tracks the true function (dashed black) and uncertainty
bands have narrowed considerably near the peak.

---
## 4. Comparing strategies: learning curves & regret

In [ ]:
# Re-run all strategies on a *fresh* simulator so histories are independent
results_clean = {}
for StrategyClass, name, kwargs in [
    (RandomStrategy,   'Random',   {'seed': 0}),
    (GridStrategy,     'Grid',     {}),
    (BayesianStrategy, 'Bayesian', {'n_init': 5, 'kappa': 2.0, 'seed': 0}),
]:
    fresh = LabSimulator(noise_std=0.12, bounds=(-2.0, 2.0), rng=42)
    strat = StrategyClass(fresh, n_experiments=N_EXPERIMENTS, **kwargs)
    xs, ys = strat.run()
    results_clean[name] = (xs, ys)

ref_sim = LabSimulator(noise_std=0.12, bounds=(-2.0, 2.0), rng=42)
fig = plot_learning_curves(results_clean, ref_sim)
plt.show()

In [ ]:
# Print summary table
ref_sim2 = LabSimulator(noise_std=0.12, bounds=(-2.0, 2.0), rng=42)
summarise(results_clean, ref_sim2)

---
## 5. Explore/Exploit: tuning kappa

The `kappa` parameter in BayesianStrategy controls the balance:
- **kappa = 0** → pure exploitation (always pick the current best estimate)
- **kappa = 5** → heavy exploration (prefer uncertain regions)

Let's see what happens:

In [ ]:
kappa_results = {}
for kappa in [0.0, 1.0, 2.0, 5.0]:
    fresh = LabSimulator(noise_std=0.12, bounds=(-2.0, 2.0), rng=42)
    strat = BayesianStrategy(fresh, n_experiments=N_EXPERIMENTS, n_init=5, kappa=kappa, seed=0)
    xs, ys = strat.run()
    kappa_results[f'κ={kappa}'] = (xs, ys)

ref_sim3 = LabSimulator(noise_std=0.12, bounds=(-2.0, 2.0), rng=42)
fig = plot_learning_curves(kappa_results, ref_sim3)
fig.suptitle('Effect of kappa on Bayesian strategy', y=1.01)
plt.tight_layout()
plt.show()

summarise(kappa_results, ref_sim3)

---
## Key takeaways

1. **Noise is manageable** — a good strategy averages it out by revisiting or
   by building a model that smooths it implicitly (like the GP).

2. **The Bayesian strategy finds the optimum with fewer experiments** than
   random or grid search because it *learns* from each experiment and focuses
   its budget on promising regions.

3. **kappa is a hyperparameter** — too low and you exploit prematurely;
   too high and you waste budget on exploration.  The sweet spot depends on
   the landscape complexity and your experiment budget.

4. **Reproducibility matters** — fixing the random seed (e.g., `rng=42`)
   means you get the same results every time you run this notebook.

---
### Suggested next steps
- Add a **2-D parameter space** (see `response_2d_gaussian_mixture` in `src/simulator.py`)
- Try **Thompson Sampling** as a fourth strategy
- Add **Expected Improvement** as an alternative acquisition function
- Run **multiple random seeds** and plot confidence bands across runs